# Test io notebook:

This notebook is used to test the implementation of **jklab-core/paths.py** module.

Testing (roughly) follows this routine:

0. import the module to test;
1. select component (variable/class/function) to test;
2. enstablish the correct behaviour;
3. implement a local function to assert the results;
4. summarise which tests have been passed.

## Module import

In [30]:
from pathlib import Path

import jklab.core.paths as jkpth

## Testing constants

In [31]:
fake_data_dir = "./test_data/paths"

## Testing helpers

In [32]:
# Collection of all test results in this notebook
test_results = {}

# Single test record function
def record_test(
    test_name,
    condition
):
    """
    Record and display the result of a test.
    """

    passed_flag = bool(condition)

    test_results[test_name] = passed_flag

    if passed_flag:
        print(f"✅ PASSED: {test_name}.")
    else:
        print(f"❌ FAILED: {test_name}.")


# Test summary function
def print_test_summary(
    test_results
):
    """
    Print a summary of test results.
    """

    # Used for visual separation
    separator_len = 40

    total = len(test_results)
    passed = sum(test_results.values())
    failed = total - passed

    print()
    print("=" * separator_len)
    print("TEST SUMMARY")
    print("=" * separator_len)

    print(f"Passed: {passed}/{total}")
    print(f"Failed: {failed}/{total}")

    if total:
        ratio = passed / total * 100
        print(f"Success rate: {ratio:.1f}%")

    print()

    for name, result in test_results.items():

        status = "PASSED" if result else "FAILED"

        print(f"{status}: {name}")

    print("=" * separator_len)


def test_raises(
    func,
    expected_error,
    expected_message=None,
):

    try:
        func()

    except expected_error as error:

        if expected_message is None:
            return True

        return expected_message in str(error)

    except Exception:
        return False

    return False

# Test 1 - validate_file()

In [33]:
def test_validate_file(file_path):

    # Validate file via pathlib
    file_path = Path(file_path)
    pathlib_flag = file_path.is_file()

    # Validate file via jklab
    jklab_flag = jkpth.validate_file(file_path)

    # Compare results
    test = (pathlib_flag == jklab_flag)

    return test

## Normal execution

In [34]:
# === Input ===
# Pick a path to a file to validate.
file_path = f"{fake_data_dir}/fake_file.dat"

# === Test & Record ===
record_test(
    test_name="validate_file: validate existing file",
    condition=test_validate_file(
        file_path=file_path
    )
)

✅ PASSED: validate_file: validate existing file.


## Validation error: missing file

In [35]:
# === Input ===
# Pick a path to a missing file.
file_path = f"{fake_data_dir}/missing_file.dat"
expected_error = FileNotFoundError

# === Test & Record ===
record_test(
    test_name="validate_file: missing file error",
    condition=test_raises(
        func=lambda: jkpth.validate_file(file_path),
        expected_error=expected_error
    )
)

✅ PASSED: validate_file: missing file error.


## Validation error: dir in place of file

In [36]:
# === Input ===
# Pick a path to a directory.
dir_path = f"{fake_data_dir}/fake_dir"
expected_error = IsADirectoryError


# === Test & Record ===
record_test(
    test_name="validate_file: is a directory error",
    condition=test_raises(
        func=lambda: jkpth.validate_file(dir_path),
        expected_error=expected_error
    )
)

✅ PASSED: validate_file: is a directory error.


# Test 2 - validate_dir()

In [37]:
def test_validate_dir(dir_path):

    # Validate dir via pathlib
    dir_path = Path(dir_path)
    pathlib_flag = dir_path.is_dir()

    # Validate file via jklab
    jklab_flag = jkpth.validate_dir(dir_path)

    # Compare results
    test = (pathlib_flag == jklab_flag)

    return test

## Normal execution

In [38]:
# === Input ===
# Pick a path to a directory to validate.
dir_path = f"{fake_data_dir}/fake_dir"

# === Test & Record ===
record_test(
    test_name="validate_dir: validate existing directory",
    condition=test_validate_dir(
        dir_path=dir_path
    )
)

✅ PASSED: validate_dir: validate existing directory.


## Validation error: missing directory

In [39]:
# === Input ===
# Pick a path to a missing directory.
dir_path = f"{fake_data_dir}/missing_dir"
expected_error = FileNotFoundError

# === Test & Record ===
record_test(
    test_name="validate_dir: missing directory error",
    condition=test_raises(
        func=lambda: jkpth.validate_dir(dir_path),
        expected_error=expected_error
    )
)

✅ PASSED: validate_dir: missing directory error.


## Validation error: file in place of a directory

In [40]:
# === Input ===
# Pick a path to a missing directory.
file_path = f"{fake_data_dir}/fake_file.dat"
expected_error = NotADirectoryError

# === Test & Record ===
record_test(
    test_name="validate_dir: is a file error",
    condition=test_raises(
        func=lambda: jkpth.validate_dir(file_path),
        expected_error=expected_error
    )
)

✅ PASSED: validate_dir: is a file error.


# Test 3 - ensure_dir()

In [41]:
def test_ensure_dir(dir_path):

    dir_path = Path(dir_path)

    # Create dir via jklab
    jkpth.ensure_dir(dir_path)

    # Check that dir exists
    test = dir_path.is_dir()

    return test

## Create directory

In [42]:
# === Input ===
# Pick a path to a directory to make.
dir_to_make = f"{fake_data_dir}/dir_that_exists"

# === Test & Record ===
record_test(
    test_name="ensure_dir: create directory",
    condition=test_ensure_dir(
        dir_path=dir_to_make
    )
)

✅ PASSED: ensure_dir: create directory.


## Create nested directory

In [43]:
# === Input ===
# Pick a path to a nested directory with a missing parent.
dir_to_make = Path(f"{fake_data_dir}/missing_parent/"
                   "existing_dir")

# === Test & Record ===
record_test(
    test_name="ensure_dir: create nested directories",
    condition=test_ensure_dir(
        dir_path=dir_to_make
    )
)

# === Cleanup ===
dir_to_make.rmdir()
dir_to_make.parent.rmdir()

✅ PASSED: ensure_dir: create nested directories.


## Existing directory

In [44]:
# === Input ===
# Pick a path to an existing
dir_to_make = f"{fake_data_dir}/fake_dir"

# === Test & Record ===
record_test(
    test_name="ensure_dir: existing directory",
    condition=test_ensure_dir(
        dir_path=dir_to_make
    )
)

✅ PASSED: ensure_dir: existing directory.


# Test 4 - list_files()

In [45]:
def test_list_files(dir_path):

    dir_path = Path(dir_path)

    # List files via pathlib
    expected_files = sorted([
        item
        for item in dir_path.iterdir()
        if item.is_file()
    ])

    # List files via jklab
    jklab_files = jkpth.list_files(dir_path)

    # Compare results
    test = (
        jklab_files
        == expected_files
    )

    return test

## List files

In [46]:
# === Input ===
# Pick a path to a directory containing files.
dir_to_check = f"{fake_data_dir}/fake_dir"

# === Test & Record ===
record_test(
    test_name=("list_files: list files "
               "and exclude directories"),
    condition=test_list_files(
        dir_path=dir_to_check
    )
)

✅ PASSED: list_files: list files and exclude directories.


## Empty directory

In [47]:
# === Input ===
# Pick a path to an empty directory.
dir_to_check = f"{fake_data_dir}/empty_dir"

# === Test & Record ===
record_test(
    test_name="list_files: empty directory",
    condition=test_list_files(
        dir_path=dir_to_check
    )
)

✅ PASSED: list_files: empty directory.


## Missing directory error

In [48]:
# === Input ===
# Pick a path to a missing directory.
dir_to_check = f"{fake_data_dir}/missing_dir"
expected_error = FileNotFoundError

# === Test & Record ===
record_test(
    test_name="list_files: missing directory",
    condition=test_raises(
        func=lambda: test_list_files(dir_to_check),
        expected_error=expected_error
    )
)

✅ PASSED: list_files: missing directory.


## Path to file error

In [49]:
# === Input ===
# Pick a path to a file.
file_to_check = f"{fake_data_dir}/fake_file.dat"
expected_error = NotADirectoryError

# === Test & Record ===
record_test(
    test_name="list_files: path to a file",
    condition=test_raises(
        func=lambda: test_list_files(file_to_check),
        expected_error=expected_error
    )
)

✅ PASSED: list_files: path to a file.


# Test 5 - list_dirs()

In [50]:
def test_list_dirs(dir_path):

    dir_path = Path(dir_path)

    # List dirs via pathlib
    expected_dirs = sorted([
        item
        for item in dir_path.iterdir()
        if item.is_dir()
    ])

    # List dirs via jklab
    jklab_dirs = jkpth.list_dirs(dir_path)

    # Compare results
    test = (
        jklab_dirs
        == expected_dirs
    )

    return test

## List directories

In [51]:
# === Input ===
# Pick a path to a directory containing directories.
dir_to_check = f"{fake_data_dir}/fake_dir"

# === Test & Record ===
record_test(
    test_name=("list_dirs: list directories "
               "and exclude files"),
    condition=test_list_dirs(
        dir_path=dir_to_check
    )
)

✅ PASSED: list_dirs: list directories and exclude files.


## Empty directory

In [52]:
# === Input ===
# Pick a path to an empty directory.
dir_to_check = f"{fake_data_dir}/empty_dir"

# === Test & Record ===
record_test(
    test_name="list_dirs: empty directory",
    condition=test_list_dirs(
        dir_path=dir_to_check
    )
)

✅ PASSED: list_dirs: empty directory.


## Missing directory error

In [53]:
# === Input ===
# Pick a path to a missing directory.
dir_to_check = f"{fake_data_dir}/missing_dir"
expected_error = FileNotFoundError

# === Test & Record ===
record_test(
    test_name="list_dirs: missing directory",
    condition=test_raises(
        func=lambda: test_list_dirs(dir_to_check),
        expected_error=expected_error
    )
)

✅ PASSED: list_dirs: missing directory.


## Path to file error

In [54]:
# === Input ===
# Pick a path to a file.
file_to_check = f"{fake_data_dir}/fake_file.dat"
expected_error = NotADirectoryError

# === Test & Record ===
record_test(
    test_name="list_dirs: path to a file",
    condition=test_raises(
        func=lambda: test_list_dirs(file_to_check),
        expected_error=expected_error
    )
)

✅ PASSED: list_dirs: path to a file.


# Summary

In [55]:
print_test_summary(test_results)


TEST SUMMARY
Passed: 17/17
Failed: 0/17
Success rate: 100.0%

PASSED: validate_file: validate existing file
PASSED: validate_file: missing file error
PASSED: validate_file: is a directory error
PASSED: validate_dir: validate existing directory
PASSED: validate_dir: missing directory error
PASSED: validate_dir: is a file error
PASSED: ensure_dir: create directory
PASSED: ensure_dir: create nested directories
PASSED: ensure_dir: existing directory
PASSED: list_files: list files and exclude directories
PASSED: list_files: empty directory
PASSED: list_files: missing directory
PASSED: list_files: path to a file
PASSED: list_dirs: list directories and exclude files
PASSED: list_dirs: empty directory
PASSED: list_dirs: missing directory
PASSED: list_dirs: path to a file
